# Module 7.6 — Alternative Runtimes & JIT

### Python Mastery · Track 7: Performance, Internals & C Extensions

The Python you usually run is **CPython**, the reference interpreter. But it is not the only way to run Python, and CPython itself is gaining just-in-time compilation. This module surveys the main alternatives and the modern speed work inside CPython, so you know what options exist when performance matters and what is coming next.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- This is a largely conceptual module: alternative interpreters such as PyPy are separate programs, so they are described rather than executed here. The runnable cells inspect the current interpreter and illustrate ideas you can observe in standard CPython.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Distinguish the major Python implementations and their trade-offs.
2. Explain what PyPy's just-in-time compiler does and when it helps.
3. Describe the JIT and specialising interpreter work in modern CPython.
4. Identify when an alternative runtime is and is not appropriate.
5. Reason about which performance approach fits a given problem.

**Prerequisites:** Tracks 1 to 6, and Modules 7.1 to 7.5.

---

## Part 1 · The Implementation Landscape

"Python" is a language specification; several programs implement it. **CPython** is the reference implementation, written in C, and what almost everyone means by "Python." Alternatives target different goals:

- **PyPy** focuses on speed through just-in-time compilation.
- **Jython** runs on the Java Virtual Machine, interoperating with Java.
- **IronPython** runs on the .NET runtime.
- **GraalPy** runs on the GraalVM, aiming for speed and polyglot interoperability.
- **MicroPython** targets microcontrollers and tiny environments.

They share the language but differ in performance, ecosystem compatibility (especially with C extensions), and target platform.

In [ ]:
import sys, platform

# Identify which implementation is running right now.
print("implementation:", sys.implementation.name)        # 'cpython' here
print("version:", platform.python_version())
print("compiler:", platform.python_compiler())

# This object distinguishes implementations programmatically.
print("implementation cache tag:", sys.implementation.cache_tag)
print()
print("On PyPy this would report 'pypy'; on Jython, 'jython', and so on.")

## Part 2 · PyPy and Just-In-Time Compilation

**PyPy** is the best-known fast alternative. Its key technique is a **just-in-time (JIT) compiler**: it begins by interpreting, watches which code runs most often (the "hot" loops), and compiles those to optimised machine code on the fly. For long-running, computation-heavy pure-Python programs, PyPy is frequently several times faster than CPython, sometimes far more.

The trade-offs matter:

- The JIT needs a **warm-up** period; very short scripts may not run long enough to benefit, and can even be slower.
- C-extension compatibility has historically been weaker than CPython's, though it has improved; libraries relying heavily on the C-API may not work or may run slower.
- It is a separate interpreter you install and run instead of CPython.

PyPy suits CPU-bound, pure-Python workloads that run long enough to warm up. It helps least for short scripts, input/output-bound work, or programs dominated by C-extension calls (where the work is already in C).

## Part 3 · The Specialising Interpreter in Modern CPython

CPython is not standing still. Recent versions added a **specialising adaptive interpreter** (the work known as PEP 659): at runtime the interpreter notices, for example, that a particular `+` always adds two integers, and swaps in a specialised, faster bytecode for that case, falling back if the assumption breaks. This happens automatically, with no code changes, and is part of why newer CPython releases are faster than older ones.

You can observe specialisation through the `dis` module's adaptive view, which shows the specialised instructions after code has run a few times. The exact instructions vary by version; the concept is what matters.

In [ ]:
import dis

def add_ints(a, b):
    return a + b

# Run it several times so the interpreter can observe the types and specialise.
for _ in range(100):
    add_ints(1, 2)

# adaptive=True shows specialised opcodes if the build has produced them.
print("disassembly with adaptive instructions:")
dis.dis(add_ints, adaptive=True)
print()
print("After repeated integer calls, the generic BINARY_OP may appear as a")
print("specialised integer-add form; this happens automatically in modern CPython.")

## Part 4 · The Experimental CPython JIT

Beyond specialisation, CPython has begun adding an actual **just-in-time compiler** of its own (introduced experimentally in version 3.13 via PEP 744, using a "copy-and-patch" technique). The aim is to compile hot code to machine code while keeping full compatibility with the existing C-extension ecosystem, which is PyPy's main limitation.

This is early-stage and opt-in (enabled with a build flag), and the speed-ups so far are modest while the foundation is laid. Combined with the free-threading work from Module 7.3, it shows CPython evolving on two fronts at once: removing the GIL for parallelism, and adding a JIT for single-thread speed. Both are multi-year efforts; the default interpreter today has neither enabled, but the direction is clear.

In [ ]:
import sys, sysconfig

# There is no portable runtime switch to query the JIT in a stable way yet,
# but we can report build information that hints at experimental features.
print("version:", sys.version.split()[0])

# The build configuration records optional features when present.
jit_flag = sysconfig.get_config_var("PY_CORE_CFLAGS")
print("this is the standard interpreter; the experimental JIT is a build-time, opt-in feature")
print("introduced in CPython 3.13+ and not enabled in a typical install")

## Part 5 · Choosing a Performance Strategy

With the whole track in view, here is how the options fit together, roughly in the order you should consider them:

1. **Measure first** (Module 7.1): find the real bottleneck before changing anything.
2. **Improve the algorithm and data structures**: the biggest wins are usually here, not in the runtime.
3. **Use better libraries**: a vectorised library such as NumPy moves hot loops into optimised C.
4. **Try an alternative runtime**: for long-running, CPU-bound, pure-Python programs with compatible dependencies, PyPy can be a near-free speed-up.
5. **Drop to C** (Module 7.5): Cython or a C extension for the hottest paths when nothing else suffices.
6. **Parallelise**: processes today (Track 5), and increasingly threads as free-threading matures (Module 7.3).

The ordering reflects effort versus reward: algorithmic and library changes are cheap and large; rewriting in C is expensive and targeted. Reach for the runtime or C only after the cheaper wins are exhausted.

In [ ]:
strategy = [
    ("1. Measure",            "profile to find the real bottleneck", "always first"),
    ("2. Algorithm",          "better complexity / data structures", "largest typical win"),
    ("3. Libraries",          "NumPy and similar move loops to C",  "cheap, big for numeric work"),
    ("4. Alternative runtime","PyPy JIT for long pure-Python jobs",  "near-free if compatible"),
    ("5. Drop to C",          "Cython or a C extension for hot paths","targeted, more effort"),
    ("6. Parallelise",        "processes now; threads as no-GIL grows","when work is parallelisable"),
]
print(f"{'step':24} {'what':40} {'when'}")
for step, what, when in strategy:
    print(f"{step:24} {what:40} {when}")

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Identifying the running implementation (Easy)

In [ ]:
import sys
print("name:", sys.implementation.name)
print("is CPython:", sys.implementation.name == "cpython")
# On a different interpreter, this name would differ (pypy, jython, ...).

### Example 2 — Reading version and build details (Easy)

In [ ]:
import platform
print("python version:", platform.python_version())
print("implementation:", platform.python_implementation())
print("build:", platform.python_build())

### Example 3 — Viewing adaptive (specialised) bytecode (Medium)

In [ ]:
import dis

def multiply(a, b):
    return a * b

for _ in range(200):
    multiply(3, 4)         # warm up with consistent integer types

dis.dis(multiply, adaptive=True)
print("repeated same-type calls let the interpreter specialise the operation")

### Example 4 — Where PyPy would and would not help (Medium)

In [ ]:
# A reasoning exercise expressed as data: which workloads suit a JIT runtime?
workloads = [
    ("long CPU-bound pure-Python loop", "PyPy helps a lot (hot loop gets compiled)"),
    ("short one-off script",            "PyPy may not warm up; little or no gain"),
    ("input/output-bound service",      "little gain; time is spent waiting, not computing"),
    ("NumPy-heavy numeric code",        "little gain; work is already in C"),
]
for workload, verdict in workloads:
    print(f"- {workload:34} -> {verdict}")

### Example 5 — Demonstrating warm-up sensitivity conceptually (Difficult)

In [ ]:
import timeit

# On a JIT runtime, the FIRST runs of a hot loop are slower (interpreting + compiling),
# and later runs are faster (running compiled code). On CPython there is no such JIT,
# so timings are steady. We illustrate by timing successive batches here.
def hot_loop(n):
    total = 0
    for i in range(n):
        total += (i * i) % 7
    return total

n = 200_000
batches = [timeit.timeit(lambda: hot_loop(n), number=3) for _ in range(4)]
print("successive batch timings on CPython (steady, no JIT warm-up):")
for i, t in enumerate(batches, 1):
    print(f"  batch {i}: {t:.4f}s")
print("on PyPy you would expect later batches to drop as the loop is compiled")

### Example 6 — A strategy decision for a scenario (Difficult)

In [ ]:
def recommend(scenario):
    """A toy recommender mapping a scenario to a performance strategy."""
    rules = {
        "slow nested loops on lists": "improve the algorithm / use sets or NumPy first",
        "long pure-python simulation": "try PyPy; or Cython the hot function",
        "numeric array crunching":     "use NumPy (vectorised C)",
        "blocking network calls":      "use async or threads, not a faster interpreter",
        "one hot function in a C app":  "Cython or a C extension for that function",
    }
    return rules.get(scenario, "measure first, then decide")

for s in ["slow nested loops on lists", "long pure-python simulation",
          "numeric array crunching", "blocking network calls"]:
    print(f"{s:30} -> {recommend(s)}")

---

## Exercises

**Exercise 1 (Easy).** Print the running implementation name and confirm whether it is CPython using `sys.implementation`.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Use the `platform` module to print the Python version and the implementation name.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Define a small function, call it many times with consistent integer arguments to warm it up, then disassemble it with `dis.dis(func, adaptive=True)` and observe the output.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Write a short list (as data) of three workloads and, for each, a one-line note on whether PyPy is likely to help and why. Print them.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Write a function `suggest(scenario)` that maps at least four performance scenarios to recommended strategies (algorithm, library, PyPy, Cython, async), and demonstrate it on several scenarios.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import sys
print("implementation:", sys.implementation.name)
print("is CPython:", sys.implementation.name == "cpython")

**Solution 2**

In [ ]:
import platform
print("version:", platform.python_version())
print("implementation:", platform.python_implementation())

**Solution 3**

In [ ]:
import dis
def f(a, b):
    return a + b
for _ in range(200):
    f(1, 2)
dis.dis(f, adaptive=True)

**Solution 4**

In [ ]:
notes = [
    ("long CPU-bound pure-Python loop", "yes: the hot loop gets JIT-compiled"),
    ("short script", "no: not enough runtime to warm up"),
    ("NumPy-heavy code", "no: the heavy work is already in C"),
]
for workload, note in notes:
    print(f"{workload}: {note}")

**Solution 5**

In [ ]:
def suggest(scenario):
    table = {
        "nested list scans": "use a set or dict (better complexity)",
        "array math": "use NumPy (vectorised C)",
        "long simulation": "try PyPy, or Cython the hot path",
        "many network calls": "use asyncio or threads",
        "one slow numeric function": "Cython or a C extension",
    }
    return table.get(scenario, "measure first")

for s in ["nested list scans", "array math", "long simulation", "many network calls"]:
    print(f"{s:24} -> {suggest(s)}")

---

## Summary and Key Points

- "Python" is a specification with several implementations: CPython (the reference), PyPy (speed via JIT), Jython (JVM), IronPython (.NET), GraalPy (GraalVM), and MicroPython (embedded); they share the language but differ in speed, C-extension compatibility, and platform.
- PyPy's just-in-time compiler compiles hot loops to machine code after a warm-up, often giving multi-fold speed-ups for long-running, CPU-bound pure-Python programs; it helps least for short scripts, input/output-bound work, or C-extension-dominated code.
- Modern CPython includes a specialising adaptive interpreter (PEP 659) that swaps in faster bytecode for observed type patterns automatically, visible via `dis.dis(..., adaptive=True)`.
- CPython has an experimental, opt-in JIT (PEP 744, 3.13+) aimed at speed without sacrificing C-extension compatibility; with free-threading, it shows CPython advancing on both parallelism and single-thread speed, though neither is enabled by default today.
- Choose a strategy by effort versus reward: measure first, then improve algorithms and data structures, use better libraries, consider an alternative runtime, drop to C for hot paths, and parallelise, in roughly that order.

### Track 7 complete

This concludes Track 7, Performance, Internals & C Extensions. You can now profile CPU and memory and optimise with discipline, understand Python's memory management and reduce footprint, reason about the object model and the GIL (and the coming free-threaded build), read bytecode and analyse the AST, interface with C through ctypes, CFFI, Cython, and hand-written extensions, and choose among runtimes and performance strategies. Track 8 turns to applied Python: building real things with databases, web services, and the data stack.